# Lesson 20 — Capstone (Intermediate)

## Goal

Implement a **piece of an optimization solver**, plug it into `discopt`, and
benchmark against the default. Three options:

### A) A new branching rule (research-style)

Implement strong branching (Lesson 14) with a custom score function, e.g. a *hybrid* of $f_j(1-f_j)$ and the pseudocost score. `discopt` 0.9 has no branching-rule option (`branching_policy=` was removed in 0.7) and no branching callback; `node_callback` only observes. You have three routes:

- **(a) Steer `discopt`'s own tree through the debugger.** `discopt.debug` lets a program attach a `DebugSession` whose frontend runs at every checkpoint. At the `BEFORE_IMPORT` checkpoint, `ctx.result_sols` holds the node relaxation points and `ctx.batch_ids` the node ids. `ctx.steer.hint(node_ids, var_indices)` then sets the branching variable for each node. Hints only *reorder* the search; they can never prune, so the certificate stays valid. Hints act only on the Python branch-and-bound path: nonconvex MINLPs, or a pure MILP with the environment variables `DISCOPT_LP_MILP_BACKEND=rust` and `DISCOPT_MILP_ENGINE=0`. The default HiGHS and Rust MILP engines have no checkpoints. A sketch:

  ```python
  from discopt import debug
  from discopt.debug.session import DebugSession
  from discopt.debug.engine import Control
  from discopt.debug.checkpoints import Checkpoint

  class MyRule:
      def interact(self, ctx, session):
          if ctx.checkpoint == Checkpoint.BEFORE_IMPORT and ctx.steer is not None:
              ids, vars_ = my_rule(ctx.batch_ids, ctx.result_sols)   # your scoring
              if ids:
                  ctx.steer.hint(ids, vars_)
          return Control.STEPI

  session = DebugSession(MyRule()); session.stop_checkpoints = set(Checkpoint)
  with debug.attach(session):
      r = m.solve()
  ```
- **(b) Your own B&B harness** (Lesson 14), with `discopt` as the reference solver.
- **(c) A fork of the Rust backend** (`crates/discopt-core/src/bnb/branching.rs`).

Document which route you took. Benchmark against the default on ≥ 5 medium MILPs (see *Instances* below).

Deliverables: rule code, results table (nodes, time, gap), and discussion.

### B) A new cut family (research-style)

Implement *zero-half cuts* (Chvátal–Gomory cuts with multipliers in $\{0, \tfrac12\}$, separated via parity arguments). Plug them in with the public cut callback: `m.solve(cut_callback=cb)`, where `cb(ctx, model)` receives a `discopt.callbacks.NodeCutContext` (the node box and `ctx.x_relaxation`) and returns a list of `discopt.callbacks.CutResult`. Every returned cut is validated against the feasible points already found, and an invalid cut raises `CutValidationError`. Only globally valid cuts are accepted. **0.9.0 caveat:** a pure MILP is routed to HiGHS or the Rust MILP engine, which never call `cut_callback`. Add `lazy_constraints=lambda ctx, model: []` to force the spatial branch-and-bound path that honours it, and use the same setting for your baseline (Lesson 13, Exercise 5). Alternatively, generate cuts *offline*, add them as constraints, and re-solve. Benchmark.

Deliverables: cut code, separation routine, results table.

### C) A new decomposition

Pick one structured problem (network design, multi-commodity flow, stochastic LP) and implement Benders or column generation from scratch. Compare runtime and gap with the monolithic solve **and** with `discopt`'s built-in decomposition where one exists: `m.solve(decomposition="benders" | "lagrangian")`, or `discopt.stochastic.solve_lshaped` (Lessons 18–19). Column generation / Dantzig–Wolfe is not built into 0.9, so a from-scratch implementation is a genuine contribution there.

## What to deliver

- `writing_response.ipynb` with code, plots, and result tables.
- `writing_response.md` (~2,000 words) describing your design,
  experiments, and conclusions.

## Scope and benchmarking protocol

Aim for **15–20 hours**. Document each design decision; cite at least 5
references; include 95 % bootstrap CIs on timing comparisons. The `discopt` source repository's
`discopt_benchmarks/utils/statistics.py` has shifted geometric means, paired comparisons and a Wilcoxon test you can copy. That directory is not part of the pip-installed package; `scipy.stats.wilcoxon` and `scipy.stats.bootstrap` work too.

**Benchmarking protocol** — apply this to every numerical comparison:

- **Instances:** at least 5 instances of *one* problem class, for a like-for-like comparison. Either generate them from a seeded random generator that you commit with your code (multi-dimensional knapsack, facility location, set cover, two-stage stochastic LP, ...), or take them from the model builders in the `discopt` source repository's `discopt_benchmarks/benchmarks/problems/` (`milp_problems.py`, `decomposition_problems.py`, ...). Report the generator, its parameters and its seeds.
- **Repetitions:** 5 runs per (instance, configuration) pair with different
  random seeds (use `--seed` if your callback uses randomness).
- **Effect-size threshold:** "method A beats method B" requires a
  Wilcoxon signed-rank test ($p < 0.05$) on per-instance log-runtime *and*
  a median speedup of $\ge 1.2\times$. Anything weaker than that is
  reported as "no significant difference."
- **Failure handling:** any instance that hits the time limit counts as the
  time limit *plus* the optimality gap reported at termination; don't drop
  unsolved instances from the average.

## References
Pick from {cite:p}`Achterberg2005,Achterberg2007,Cornuejols2008,Benders1962,DantzigWolfe1960,Wolsey1998` based on your project.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm
